# CFPB Consumer Complaints Analysis

**Dataset:** [Consumer Complaint Database](https://www.kaggle.com/datasets/selener/consumer-complaint-database) — U.S. Consumer Financial Protection Bureau (CFPB)  
**Coverage:** December 2011 – May 2019 · ~1.28 million records · 18 fields  
**Tools:** Python · Polars · Plotly  

---

This notebook covers data loading, quality assessment, cleaning, and exploratory analysis of U.S. consumer financial complaints. The cleaned dataset and pre-aggregated summary tables are exported to `dashboard_data/` for use in a BI dashboard.

**Analytical focus:**
- Which products and issues drive the most complaints, and how has volume evolved over time?
- Which companies attract the most consumer friction?
- How do response quality and dispute rates differ by product?
- Where are the data quality limitations that could affect dashboard interpretation?

## 1. Setup & Data Loading

All dependencies are imported here. `kagglehub` downloads the dataset on the first run and caches it locally — subsequent runs skip the download.

In [41]:
import os
import kagglehub
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Download dataset — cached locally after first run
path = kagglehub.dataset_download("selener/consumer-complaint-database")
print("Dataset path:", path)

Dataset path: /Users/stefanbozhilov/.cache/kagglehub/datasets/selener/consumer-complaint-database/versions/1


In [42]:
csv_file = next(f for f in os.listdir(path) if f.endswith(".csv"))
df = pl.read_csv(os.path.join(path, csv_file))
df = df.with_columns(
    pl.col("Date received").str.to_date(format="%m/%d/%Y"),
    pl.col("Date sent to company").str.to_date(format="%m/%d/%Y"),
).sort("Date received")

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded: 1,282,355 rows × 18 columns


## 2. Data Overview & Quality Assessment

Before cleaning, we inspect the schema and field completeness to identify issues that need to be addressed.

In [43]:
def pl_print(df: pl.DataFrame, tbl_rows: int | None = None, tbl_cols: int | None = None) -> None:
    """Display a Polars DataFrame without row/column truncation."""
    with pl.Config(
        tbl_rows=tbl_rows if tbl_rows is not None else df.shape[0],
        tbl_cols=tbl_cols if tbl_cols is not None else df.shape[1],
    ):
        print(df)

In [44]:
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
print(f"{'Column':<35} {'Type':<15} {'Nulls':>10}  {'Null %':>7}")
print("-" * 74)
for col, dtype in zip(df.columns, df.dtypes):
    n_null = df[col].null_count()
    pct = n_null / df.shape[0] * 100
    print(f"  {col:<33} {str(dtype):<15} {n_null:>10,}  {pct:>6.1f}%")

Shape: 1,282,355 rows × 18 columns

Column                              Type                 Nulls   Null %
--------------------------------------------------------------------------
  Date received                     Date                     0     0.0%
  Product                           String                   0     0.0%
  Sub-product                       String             235,166    18.3%
  Issue                             String                   0     0.0%
  Sub-issue                         String             531,186    41.4%
  Consumer complaint narrative      String             898,791    70.1%
  Company public response           String             833,273    65.0%
  Company                           String                   0     0.0%
  State                             String              19,400     1.5%
  ZIP code                          String             115,298     9.0%
  Tags                              String           1,106,712    86.3%
  Consumer consent provid

### 2.1 Missing Values

Most core fields are fully populated. Notable gaps:

- **Consumer complaint narrative** (70% missing) — consumers must opt in to share their narrative publicly. This restricts text-based analysis to a minority of records and introduces selection bias.
- **Tags** (86% missing) — indicates servicemember or older-adult status; rarely populated and not suitable for segmentation at scale.
- **Sub-issue** (41%) and **Sub-product** (18%) — hierarchical refinements not always provided; use with caution in granular breakdowns.
- **Company public response** (65%) — voluntary; absence does not imply a negative outcome.

In [45]:
missing = (df.null_count() / df.shape[0] * 100).transpose(include_header=True, column_names=["missing_pct"])
pl_print(missing)

shape: (18, 2)
┌──────────────────────────────┬─────────────┐
│ column                       ┆ missing_pct │
│ ---                          ┆ ---         │
│ str                          ┆ f64         │
╞══════════════════════════════╪═════════════╡
│ Date received                ┆ 0.0         │
│ Product                      ┆ 0.0         │
│ Sub-product                  ┆ 18.338604   │
│ Issue                        ┆ 0.0         │
│ Sub-issue                    ┆ 41.422695   │
│ Consumer complaint narrative ┆ 70.089094   │
│ Company public response      ┆ 64.9799     │
│ Company                      ┆ 0.0         │
│ State                        ┆ 1.512842    │
│ ZIP code                     ┆ 8.991114    │
│ Tags                         ┆ 86.303091   │
│ Consumer consent provided?   ┆ 2.303106    │
│ Submitted via                ┆ 0.0         │
│ Date sent to company         ┆ 0.0         │
│ Company response to consumer ┆ 0.000546    │
│ Timely response?             ┆ 0.0         

## 3. Data Cleaning & Feature Engineering

Key steps applied before analysis:

- **Product normalisation** — all product variants (legacy and current CFPB names) are consolidated to short, consistent display labels using prefix matching, which is immune to subtle string differences across dataset versions.
- **Response time** — `response_days` is derived from the gap between `Date received` and `Date sent to company`.
- **Temporal features** — `year` and `year_month` columns are added for time-series aggregations.
- **Dispute field cleanup** — `"N/A"` values in `Consumer disputed?` (introduced when the field was deprecated in 2017) are replaced with `null`.
- **Deduplication** — rows sharing a `Complaint ID` are deduplicated, keeping the first occurrence.

In [63]:
# Show the renaming problem: filter to only the affected categories and pivot
# by year so you can see exactly when each name appeared and disappeared.
# Note: year is extracted inline here because this runs on the raw df.
renamed = (
    df
    .with_columns(pl.col("Date received").dt.year().alias("year"))
    .filter(
        pl.col("Product").str.starts_with("Credit reporting") |
        pl.col("Product").str.starts_with("Credit card") |
        pl.col("Product").str.starts_with("Prepaid card") |
        pl.col("Product").str.starts_with("Bank account") |
        pl.col("Product").str.starts_with("Checking or savings") |
        pl.col("Product").str.starts_with("Payday loan") |
        pl.col("Product").str.starts_with("Money transfer") |
        pl.col("Product").str.starts_with("Virtual currency")
    )
    .group_by(["Product", "year"])
    .agg(pl.len().alias("n"))
)

pivot = (
    renamed
    .pivot(on="year", index="Product", values="n")
    .fill_null(0)
    .sort("Product")
)

# Sort year columns chronologically (pivot returns them in arbitrary order)
year_cols = sorted([c for c in pivot.columns if c != "Product"], key=int)
pl_print(pivot.select(["Product"] + year_cols))

shape: (12, 10)
┌──────────────────────────┬──────┬───────┬───────┬───────┬───────┬───────┬───────┬────────┬───────┐
│ Product                  ┆ 2011 ┆ 2012  ┆ 2013  ┆ 2014  ┆ 2015  ┆ 2016  ┆ 2017  ┆ 2018   ┆ 2019  │
│ ---                      ┆ ---  ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---    ┆ ---   │
│ str                      ┆ u32  ┆ u32   ┆ u32   ┆ u32   ┆ u32   ┆ u32   ┆ u32   ┆ u32    ┆ u32   │
╞══════════════════════════╪══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪════════╪═══════╡
│ Bank account or service  ┆ 0    ┆ 12212 ┆ 13388 ┆ 14662 ┆ 17140 ┆ 21848 ┆ 6956  ┆ 0      ┆ 0     │
│ Checking or savings      ┆ 0    ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 12764 ┆ 21212  ┆ 6665  │
│ account                  ┆      ┆       ┆       ┆       ┆       ┆       ┆       ┆        ┆       │
│ Credit card              ┆ 1260 ┆ 15353 ┆ 13105 ┆ 13974 ┆ 17300 ┆ 21065 ┆ 7133  ┆ 0      ┆ 0     │
│ Credit card or prepaid   ┆ 0    ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆

In [47]:
# Product normalisation using prefix matching — immune to exact-string differences
# between legacy and current CFPB category names.
normalise_product = (
    pl.when(pl.col("Product").str.starts_with("Credit reporting"))
      .then(pl.lit("Credit Reporting"))
    .when(
        pl.col("Product").str.starts_with("Credit card") |
        pl.col("Product").str.starts_with("Prepaid card")
    )
      .then(pl.lit("Credit Card"))
    .when(
        pl.col("Product").str.starts_with("Bank account") |
        pl.col("Product").str.starts_with("Checking or savings")
    )
      .then(pl.lit("Bank Account"))
    .when(pl.col("Product").str.starts_with("Payday loan"))
      .then(pl.lit("Payday / Personal Loan"))
    .when(
        pl.col("Product").str.starts_with("Money transfer") |
        pl.col("Product").str.starts_with("Virtual currency")
    )
      .then(pl.lit("Money Transfer"))
    .otherwise(pl.col("Product"))
    .alias("Product")
)

df_clean = (
    df
    .with_columns(
        normalise_product,
        (pl.col("Date sent to company") - pl.col("Date received"))
          .dt.total_days()
          .alias("response_days"),
        pl.col("Date received").dt.year().cast(pl.Int32).alias("year"),
        pl.col("Date received").dt.strftime("%Y-%m").alias("year_month"),
        pl.col("Consumer disputed?").replace({"N/A": None}),
    )
    .unique(subset=["Complaint ID"], keep="first")
    .sort("Date received")
)

print(f"Rows before dedup: {df.shape[0]:,}")
print(f"Rows after dedup:  {df_clean.shape[0]:,}")
print(f"Product categories after normalisation ({df_clean['Product'].n_unique()}):")
print(df_clean["Product"].value_counts(sort=True))

Rows before dedup: 1,282,355
Rows after dedup:  1,282,355
Product categories after normalisation (11):
shape: (11, 2)
┌─────────────────────────┬────────┐
│ Product                 ┆ count  │
│ ---                     ┆ ---    │
│ str                     ┆ u32    │
╞═════════════════════════╪════════╡
│ Credit Reporting        ┆ 366410 │
│ Mortgage                ┆ 278098 │
│ Debt collection         ┆ 244873 │
│ Credit Card             ┆ 140662 │
│ Bank Account            ┆ 126847 │
│ …                       ┆ …      │
│ Consumer Loan           ┆ 31605  │
│ Money Transfer          ┆ 15536  │
│ Payday / Personal Loan  ┆ 14203  │
│ Vehicle loan or lease   ┆ 11377  │
│ Other financial service ┆ 1059   │
└─────────────────────────┴────────┘


## 4. Exploratory Data Analysis

The analysis covers four areas: distribution of key categoricals, response quality and dispute rates by product, top companies by volume, narrative availability, and complaint volume trends over time. All charts use `df_clean`.

In [48]:
cols = ["Product", "Issue", "Submitted via", "Company response to consumer"]
color = "#4C6EF5"

fig = make_subplots(rows=2, cols=2, subplot_titles=cols, horizontal_spacing=0.18, vertical_spacing=0.12)

for (row, col), field in zip([(1,1),(1,2),(2,1),(2,2)], cols):
    vc = df_clean[field].fill_null("(null)").value_counts(sort=True).head(10)
    labels = [l[:40] for l in vc[field].to_list()[::-1]]
    counts = vc["count"].to_list()[::-1]
    bar_colors = ["#8D1515" if l == "(null)" else color for l in labels]
    fig.add_trace(go.Bar(x=counts, y=labels, orientation='h', marker_color=bar_colors, showlegend=False), row=row, col=col)

fig.update_layout(
    height=800, width=1200,
    title_text="4.1 — Complaint Distributions: Product, Issue, Channel & Company Response",
    title_font_size=16,
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=11),
    margin=dict(l=20, r=20, t=60, b=20),
)
fig.update_xaxes(showgrid=True, gridcolor="#eeeeee", zeroline=False)
fig.update_yaxes(tickfont=dict(size=10))
fig.show()

### 4.2 Response Quality & Dispute Rates

Products with high **late response rates** signal operational capacity issues at the company or CFPB processing level. Products with high **consumer dispute rates** indicate unresolved or unsatisfactory outcomes — a useful proxy for consumer pain and operational risk.

> Note: the `Consumer disputed?` field was deprecated after 2017, so dispute rates are calculated only on records where the field is populated.

In [49]:
risk = (
    df_clean
    .group_by("Product")
    .agg([
        (pl.col("Timely response?") == "No").mean().alias("late_response_rate"),
        (pl.col("Consumer disputed?") == "Yes").mean().fill_null(0).alias("dispute_rate"),
    ])
    .sort("dispute_rate", descending=True)
)

products = [p[:35] for p in risk["Product"].to_list()]
late    = [round(v * 100, 1) for v in risk["late_response_rate"].to_list()]
dispute = [round(v * 100, 1) for v in risk["dispute_rate"].to_list()]

fig = go.Figure()
fig.add_trace(go.Bar(name="Late response %",    x=products, y=late,    marker_color="#4C6EF5"))
fig.add_trace(go.Bar(name="Consumer dispute %", x=products, y=dispute, marker_color="#E03131"))

fig.update_layout(
    barmode="group",
    title="Late Response & Dispute Rate by Product",
    yaxis_title="Rate (%)",
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=11),
    legend=dict(orientation="h", y=1.08),
    margin=dict(l=20, r=20, t=60, b=120),
    xaxis_tickangle=-35,
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor="#eeeeee", zeroline=False)
fig.show()

### 4.3 Top Companies by Complaint Volume

The three credit bureaus (Equifax, Experian, TransUnion) account for a disproportionate share of complaints, reflecting both their market reach and the high volume of credit-reporting disputes. Among banks, complaint counts broadly track market share.

In [50]:
pl_print(
    df_clean
    .group_by("Company")
    .agg(pl.len().alias("complaints"))
    .sort("complaints", descending=True)
    .head(20)
)

shape: (20, 2)
┌─────────────────────────────────┬────────────┐
│ Company                         ┆ complaints │
│ ---                             ┆ ---        │
│ str                             ┆ u32        │
╞═════════════════════════════════╪════════════╡
│ EQUIFAX, INC.                   ┆ 115703     │
│ Experian Information Solutions… ┆ 103784     │
│ TRANSUNION INTERMEDIATE HOLDIN… ┆ 96587      │
│ BANK OF AMERICA, NATIONAL ASSO… ┆ 82104      │
│ WELLS FARGO & COMPANY           ┆ 70919      │
│ JPMORGAN CHASE & CO.            ┆ 60221      │
│ CITIBANK, N.A.                  ┆ 49058      │
│ CAPITAL ONE FINANCIAL CORPORAT… ┆ 34581      │
│ Navient Solutions, LLC.         ┆ 29296      │
│ OCWEN LOAN SERVICING LLC        ┆ 27750      │
│ SYNCHRONY FINANCIAL             ┆ 21925      │
│ NATIONSTAR MORTGAGE             ┆ 20449      │
│ U.S. BANCORP                    ┆ 17120      │
│ Ditech Financial LLC            ┆ 14104      │
│ AMERICAN EXPRESS COMPANY        ┆ 13747      │
│ PNC

### 4.4 Narrative Availability by Product

Only ~30% of complaints include a free-text consumer narrative. Availability varies by product. Any NLP or text-mining use case will be working with a non-random subset of complaints — consumers who opted in may differ systematically from those who did not.

In [51]:
narrative = (
    df_clean
    .with_columns((pl.col("Consumer complaint narrative").is_not_null()).alias("has_narrative"))
    .group_by("Product")
    .agg(pl.col("has_narrative").mean())
    .sort("has_narrative", descending=True)
)

products = [p[:35] for p in narrative["Product"].to_list()]
rates    = [round(v * 100, 1) for v in narrative["has_narrative"].to_list()]
colors   = ["#2F9E44" if v >= 40 else "#4C6EF5" if v >= 20 else "#E03131" for v in rates]

fig = go.Figure(go.Bar(
    x=rates, y=products, orientation='h',
    marker_color=colors,
    text=[f"{v}%" for v in rates],
    textposition="outside",
))
fig.update_layout(
    title="4.4 — Narrative Availability by Product (% of complaints with consumer narrative)",
    xaxis_title="% with narrative",
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=11),
    margin=dict(l=20, r=80, t=60, b=20),
    xaxis=dict(range=[0, 120], showgrid=True, gridcolor="#eeeeee"),
    yaxis=dict(automargin=True),
)
fig.show()

### 4.5 Complaint Volume Over Time

Complaint volume has grown consistently every year since the CFPB began accepting complaints in late 2011. The 2011 figure (2,536) reflects only a partial year of operation. From the first full year (2012: ~72k) through 2018 (~257k), volume roughly tripled — likely driven by growing consumer awareness of the CFPB, an expanding product scope, and easier web submission. The 2019 figure (~86k) covers January–May only and is not comparable to full-year totals.

In [ ]:
yearly = (
    df_clean
    .group_by("year")
    .agg(pl.len().alias("complaints"))
    .sort("year")
)

years  = yearly["year"].to_list()
counts = yearly["complaints"].to_list()
cut    = years.index(2019)  # index where the partial year starts

fig = go.Figure()

# Solid filled area for complete years (2011-2018)
fig.add_trace(go.Scatter(
    x=years[:cut], y=counts[:cut],
    mode="lines+markers",
    line=dict(color="#4C6EF5", width=2.5),
    marker=dict(size=8, color="#4C6EF5"),
    fill="tozeroy",
    fillcolor="rgba(76, 110, 245, 0.08)",
    showlegend=False,
))

# Dashed bridge from 2018 to 2019 to signal the partial year
fig.add_trace(go.Scatter(
    x=[years[cut - 1], years[cut]],
    y=[counts[cut - 1], counts[cut]],
    mode="lines+markers",
    line=dict(color="#868E96", width=2, dash="dot"),
    marker=dict(size=8, color="#868E96", symbol="circle-open"),
    showlegend=False,
))

fig.add_annotation(
    x=2019, y=counts[cut],
    text="Jan–May only",
    showarrow=True, arrowhead=2, arrowcolor="#868E96",
    ax=55, ay=-40,
    font=dict(color="#868E96", size=11),
)

fig.update_layout(
    title="4.5 — Annual Complaint Volume (2011–2019)",
    xaxis_title="Year",
    yaxis_title="Number of Complaints",
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=12),
    xaxis=dict(showgrid=False, dtick=1),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee", zeroline=False),
    margin=dict(l=20, r=20, t=60, b=40),
)

fig.show()

## 5. Dashboard Export

The cleaned dataset and three pre-aggregated summary tables are written to `dashboard_data/` as Parquet files for direct consumption by the BI tool.

| File | Contents |
|------|----------|
| `cfpb_complaints_clean.parquet` | Full cleaned dataset with all engineered columns |
| `complaints_by_product_year.parquet` | Complaint counts by product × year |
| `company_response_quality.parquet` | Late-response and dispute rates by company × product |

In [54]:
out_dir = "dashboard_data"
os.makedirs(out_dir, exist_ok=True)

# Full cleaned dataset
df_clean.write_parquet(f"{out_dir}/cfpb_complaints_clean.parquet")

# Complaints by product × year — for trend charts
(
    df_clean
    .group_by(["Product", "year"])
    .agg(pl.len().alias("complaints"))
    .sort(["Product", "year"])
    .write_parquet(f"{out_dir}/complaints_by_product_year.parquet")
)

# Company response quality — for response/dispute dashboards
(
    df_clean
    .group_by(["Company", "Product"])
    .agg([
        pl.len().alias("total_complaints"),
        (pl.col("Timely response?") == "No").mean().alias("late_response_rate"),
        (pl.col("Consumer disputed?") == "Yes").mean().alias("dispute_rate"),
    ])
    .sort("total_complaints", descending=True)
    .write_parquet(f"{out_dir}/company_response_quality.parquet")
)

print(f"Exported {df_clean.shape[0]:,} rows to '{out_dir}/':")
for fname in sorted(os.listdir(out_dir)):
    size_kb = os.path.getsize(f"{out_dir}/{fname}") / 1024
    print(f"  {fname}  ({size_kb:.0f} KB)")

Exported 1,282,355 rows to 'dashboard_data/':
  cfpb_complaints_clean.parquet  (129502 KB)
  company_response_quality.parquet  (112 KB)
  complaints_by_product_year.parquet  (2 KB)
